# Tutorial: Alocação de Leitos de UTI com Timeout

**Público:** alunos que já entendem fila simples e prioridade.

**Pré-requisitos:** saber o que é `Process` e ter noção da diferença entre capacidade e objeto.

**Objetivos de aprendizagem:**

- entender por que leito nomeado pede `Store` e não `Resource`;
- combinar espera por leito com prazo máximo;
- interpretar o operador `|` como "um evento ou outro".


## Roteiro

1. Entender o cenário da UTI.
2. Decidir a estrutura correta para representar leitos.
3. Modelar o timeout de espera.
4. Rodar a simulação.
5. Interpretar quem internou e quem foi transferido.


In [1]:
from __future__ import annotations

import simpy

print(f"Versão do SimPy: {simpy.__version__}")

Versão do SimPy: 4.1.1


## 1. Cenário

Temos dois leitos identificáveis:

- `UTI-A`
- `UTI-B`

Pacientes chegam ao longo do tempo e pedem um leito.

Se o leito não aparecer antes do prazo máximo de espera, o paciente é transferido.

Esse prazo é um ponto didático importante: ele mostra como modelar uma espera que **não pode durar para sempre**.


## 2. Conceito fundamental: por que `Store`?

Aqui o leito não é apenas "uma unidade de capacidade".

Nós queremos saber **qual leito específico foi ocupado**.

Por isso usamos `Store`.

Se usássemos `Resource`, saberíamos apenas que existe capacidade ocupada, mas não teríamos a identidade do item (`UTI-A` ou `UTI-B`).


In [2]:
PACIENTES = [
    ("P101", 0, 10, 7),
    ("P102", 1, 12, 6),
    ("P103", 4, 8, 7),
    ("P104", 5, 6, 4),
]

PACIENTES

[('P101', 0, 10, 7), ('P102', 1, 12, 6), ('P103', 4, 8, 7), ('P104', 5, 6, 4)]

### Estrutura dos dados

Cada paciente tem:

- nome;
- instante de chegada;
- tempo de permanência na UTI;
- tempo máximo de espera por um leito.


In [3]:
def internacao_uti(env, nome, chegada, permanencia, espera_max, leitos):
    yield env.timeout(chegada)
    print(f"{env.now:02.0f} h | {nome} solicita leito de UTI")

    pedido = leitos.get()
    resultado = yield pedido | env.timeout(espera_max)

    if pedido in resultado:
        leito = resultado[pedido]
        print(f"{env.now:02.0f} h | {nome} ocupa {leito}")
        yield env.timeout(permanencia)
        print(f"{env.now:02.0f} h | {nome} recebe alta do {leito}")
        yield leitos.put(leito)
    else:
        pedido.cancel()
        print(f"{env.now:02.0f} h | {nome} não conseguiu leito e é transferido")

## 3. Linha mais importante do notebook

A linha abaixo concentra o conceito central:

`resultado = yield pedido | env.timeout(espera_max)`

Leia assim:

> "o processo vai esperar até acontecer uma destas coisas: aparecer um leito ou acabar o prazo".

Isso é muito útil em saúde, logística e atendimento com SLA.


In [4]:
def executar_simulacao():
    env = simpy.Environment()
    leitos = simpy.Store(env, capacity=2)
    leitos.items.extend(["UTI-A", "UTI-B"])

    for dados in PACIENTES:
        env.process(internacao_uti(env, *dados, leitos))

    env.run()


executar_simulacao()

00 h | P101 solicita leito de UTI
00 h | P101 ocupa UTI-A
01 h | P102 solicita leito de UTI
01 h | P102 ocupa UTI-B
04 h | P103 solicita leito de UTI
05 h | P104 solicita leito de UTI
09 h | P104 não conseguiu leito e é transferido
10 h | P101 recebe alta do UTI-A
10 h | P103 ocupa UTI-A
13 h | P102 recebe alta do UTI-B
18 h | P103 recebe alta do UTI-A


## 4. O que observar na saída

Pontos principais:

- `P101` e `P102` ocupam os dois leitos iniciais.
- `P103` e `P104` ficam na espera.
- `P104` vence no relógio do timeout antes de aparecer um leito.
- `P103` ainda consegue internar quando `UTI-A` é liberado.

A leitura operacional é clara:

**não basta ter fila; é preciso saber se a fila pode esperar**.


## 5. Erro comum

Se você esquecer `pedido.cancel()` quando o timeout vence, pode deixar um pedido pendurado na fila do `Store`.

Didaticamente, isso é um bom ponto para mostrar que o modelo também precisa de higiene de eventos.


## 6. Exercícios

1. Aumente o prazo máximo de `P104` para `6` horas. Ele ainda é transferido?
2. Adicione um terceiro leito no início da simulação.
3. Explique por que este problema não é bem representado por `Container`.


In [5]:
# Espaço para experimentos:
# - altere a lista PACIENTES;
# - altere a lista inicial de leitos;
# - rode novamente a simulação.

## 7. Extensão sugerida

Em uma versão mais realista, você pode incluir:

- tipo de isolamento;
- compatibilidade clínica;
- prioridade por gravidade;
- origem regulada externa.

Aí a escolha natural costuma evoluir de `Store` para `FilterStore`.
